# FFT-accelerated beta-2 (L2) divergence experiments

> **Optional/legacy notebook.** The supported main experiments now use exact Gaussian formulas or direct spatial quadrature. This notebook retains the earlier periodic Fourier experiments and now contains only experiments based on the beta-2 divergence.

For the density-power-divergence convention, beta equal to 2 gives half the squared L2 distance:

$$
D_2(p,q)=\frac{1}{2}\int_{-\infty}^{\infty}(p(x)-q(x))^2\,\mathrm{d}x.
$$

For Gaussian KDEs based on the same sample, define the exact Gram matrix

$$
G_{ij}=\int \widehat f_{h_i}(x)\widehat f_{h_j}(x)\,\mathrm{d}x.
$$

Then

$$
D_2(\widehat f_{h_i},\widehat f_{h_j})
=\frac{1}{2}\left(G_{ii}+G_{jj}-2G_{ij}\right).
$$

The analytic Gaussian-KDE formula is exact but costs $O(N^2n^2)$. The experiments below instead evaluate the whole KDE family once by FFT and assemble $G$ with the matrix product $G \approx \Delta x\,FF^T$. The analytic helper is retained for the small validation check below. Control the FFT approximation with `beta2_half_width` and `beta2_spatial_points`.


In [35]:
from __future__ import annotations

import importlib

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from kde_tools.bandwidths import (
    gaussian_mise_oracle_bandwidth,
    nearest_bandwidth_grid_index,
)
from kde_tools.fourier import (
    beta2_divergence_matrix_fft,
    beta2_similarity_matrix_fft,
)
import kde_tools.similarities as kde_similarities

# Reload helpers that may have changed since this kernel was started.
importlib.reload(kde_similarities)
from kde_tools.similarities import (
    beta2_divergence_matrix,
    doob_stationary_mass,
    first_order_perron_vector,
    perron_vector,
    population_gaussian_beta2_divergence_matrix,
    toeplitz_corrected_choice,
)

np.set_printoptions(precision=5, suppress=True)
seed = 20260818


## 1. Example stationary distributions on geometric and linear grids

Choose the sample size, number of bandwidth candidates, bandwidth interval, and similarity scale below. One random sample is reused for both grids. The first figure compares the uncorrected Doob stationary masses. For symmetric beta-2 similarities, stationary mass is proportional to the squared Perron vector; consequently, the second figure normalizes the squared Toeplitz-correction score and plots the resulting stationary-mass ratio as a probability distribution.


In [37]:
stationary_example_seed = seed + 5
stationary_example_n = 2000
stationary_example_N = 101
stationary_example_h_min = 0.01
stationary_example_h_max = 2.0
stationary_example_tau = 1.0
stationary_example_half_width = 12.0
stationary_example_spatial_points = 8192

if stationary_example_n < 1:
    raise ValueError("stationary_example_n must be positive.")
if stationary_example_N < 2:
    raise ValueError("stationary_example_N must be at least two.")
if not 0.0 < stationary_example_h_min < stationary_example_h_max:
    raise ValueError("Require 0 < h_min < h_max.")
if stationary_example_tau <= 0.0:
    raise ValueError("stationary_example_tau must be positive.")
if stationary_example_half_width <= 0.0:
    raise ValueError("stationary_example_half_width must be positive.")
if (
    stationary_example_spatial_points < 2
    or stationary_example_spatial_points % 2
):
    raise ValueError(
        "stationary_example_spatial_points must be a positive even integer."
    )

stationary_example_grids = {
    "geometric grid": np.geomspace(
        stationary_example_h_min,
        stationary_example_h_max,
        stationary_example_N,
    ),
    "linear grid": np.linspace(
        stationary_example_h_min,
        stationary_example_h_max,
        stationary_example_N,
    ),
}
stationary_example_rng = np.random.default_rng(stationary_example_seed)
stationary_example_samples = stationary_example_rng.normal(
    size=stationary_example_n
)
stationary_example_results = {}

for grid_label, bandwidth_grid in stationary_example_grids.items():
    similarity = beta2_similarity_matrix_fft(
        samples=stationary_example_samples,
        h_grid=bandwidth_grid,
        tau=stationary_example_tau,
        half_width=stationary_example_half_width,
        spatial_points=stationary_example_spatial_points,
    )
    uncorrected_mass = doob_stationary_mass(similarity)
    _, _, correction_score = toeplitz_corrected_choice(
        similarity=similarity,
        bandwidths=bandwidth_grid,
    )
    corrected_mass = correction_score**2
    corrected_mass /= corrected_mass.sum()
    stationary_example_results[grid_label] = {
        "bandwidths": bandwidth_grid,
        "uncorrected_mass": uncorrected_mass,
        "corrected_mass": corrected_mass,
    }

stationary_example_colors = {
    "geometric grid": "#1f77b4",
    "linear grid": "#0b3c5d",
}

stationary_example_uncorrected_figure = go.Figure()
for grid_label, result in stationary_example_results.items():
    stationary_example_uncorrected_figure.add_trace(
        go.Scatter(
            x=result["bandwidths"],
            y=result["uncorrected_mass"],
            mode="lines+markers",
            line=dict(color=stationary_example_colors[grid_label], width=2.5),
            marker=dict(size=5),
            name=grid_label,
            hovertemplate=(
                f"{grid_label}"
                "<br>h=%{x:.6g}"
                "<br>stationary mass=%{y:.6g}<extra></extra>"
            ),
        )
    )
for grid_label, result in stationary_example_results.items():
    peak_index = int(np.argmax(result["uncorrected_mass"]))
    peak_h = float(result["bandwidths"][peak_index])
    stationary_example_uncorrected_figure.add_vline(
        x=peak_h,
        line=dict(color="#d62728", width=2, dash="dash"),
        annotation_text=f"{grid_label} peak: h={peak_h:.4g}",
        annotation_position=(
            "top left" if grid_label == "geometric grid" else "top right"
        ),
        annotation_font_color="#d62728",
    )
stationary_example_uncorrected_figure.update_xaxes(
    title_text="bandwidth h",
    range=[
        stationary_example_h_min,
        stationary_example_h_max,
    ],
)
stationary_example_uncorrected_figure.update_yaxes(
    title_text="uncorrected Doob stationary mass",
    rangemode="tozero",
)
stationary_example_uncorrected_figure.update_layout(
    title=(
        "Uncorrected beta-2 stationary distributions: "
        f"n={stationary_example_n}, N={stationary_example_N}"
    ),
    width=1050,
    height=550,
    template="plotly_white",
    legend=dict(orientation="h", x=0.5, xanchor="center", y=-0.18),
    margin=dict(t=100, b=120),
)
stationary_example_uncorrected_figure.show()

stationary_example_corrected_figure = go.Figure()
for grid_label, result in stationary_example_results.items():
    stationary_example_corrected_figure.add_trace(
        go.Scatter(
            x=result["bandwidths"],
            y=result["corrected_mass"],
            mode="lines+markers",
            line=dict(color=stationary_example_colors[grid_label], width=2.5),
            marker=dict(size=5),
            name=grid_label,
            hovertemplate=(
                f"{grid_label}"
                "<br>h=%{x:.6g}"
                "<br>corrected stationary mass=%{y:.6g}<extra></extra>"
            ),
        )
    )
for grid_label, result in stationary_example_results.items():
    peak_index = int(np.argmax(result["corrected_mass"]))
    peak_h = float(result["bandwidths"][peak_index])
    stationary_example_corrected_figure.add_vline(
        x=peak_h,
        line=dict(color="#d62728", width=2, dash="dash"),
        annotation_text=f"{grid_label} peak: h={peak_h:.4g}",
        annotation_position=(
            "top left" if grid_label == "geometric grid" else "top right"
        ),
        annotation_font_color="#d62728",
    )
stationary_example_corrected_figure.update_xaxes(
    title_text="bandwidth h",
    range=[
        stationary_example_h_min,
        stationary_example_h_max,
    ],
)
stationary_example_corrected_figure.update_yaxes(
    title_text="normalized Toeplitz-corrected stationary mass",
    rangemode="tozero",
)
stationary_example_corrected_figure.update_layout(
    title=(
        "Toeplitz-corrected beta-2 stationary distributions: "
        f"n={stationary_example_n}, N={stationary_example_N}"
    ),
    width=1050,
    height=550,
    template="plotly_white",
    legend=dict(orientation="h", x=0.5, xanchor="center", y=-0.18),
    margin=dict(t=100, b=120),
)
stationary_example_corrected_figure.show()


## 2. Configurable beta-2 selection histograms

Choose $n$, $N$, $M$, the bandwidth interval, and $\tau$ below. The same Monte Carlo samples are used for the uncorrected and Toeplitz-corrected selections. The two distributions are displayed in separate histograms.

In [38]:
beta2_n = 200
beta2_N = 201
beta2_M = 100

beta2_h_min = 0.01
beta2_h_max = 2.0
beta2_tau = 1.0
beta2_seed = seed + 10
beta2_half_width = 12.0
beta2_spatial_points = 8192
beta2_validate_fft = True

if beta2_n < 1 or beta2_M < 1:
    raise ValueError("beta2_n and beta2_M must be positive.")
if beta2_N < 2:
    raise ValueError("beta2_N must be at least 2.")
if not 0.0 < beta2_h_min < beta2_h_max:
    raise ValueError("Require 0 < beta2_h_min < beta2_h_max.")
if beta2_tau <= 0.0:
    raise ValueError("beta2_tau must be positive.")
if beta2_half_width <= 0.0:
    raise ValueError("beta2_half_width must be positive.")
if beta2_spatial_points < 2:
    raise ValueError("beta2_spatial_points must be at least 2.")

beta2_h_grid = np.geomspace(
    beta2_h_min,
    beta2_h_max,
    beta2_N,
)

print(
    f"beta=2, n={beta2_n}, N={beta2_N}, M={beta2_M}, "
    f"tau={beta2_tau:g}"
)

beta=2, n=200, N=201, M=100, tau=1


In [39]:
if beta2_validate_fft:
    validation_rng = np.random.default_rng(beta2_seed)
    validation_samples = validation_rng.normal(
        size=min(beta2_n, 100)
    )
    validation_indices = np.unique(
        np.linspace(0, beta2_N - 1, min(beta2_N, 20), dtype=int)
    )
    validation_h_grid = beta2_h_grid[validation_indices]
    validation_exact = beta2_divergence_matrix(
        samples=validation_samples,
        h_grid=validation_h_grid,
    )
    validation_fft = beta2_divergence_matrix_fft(
        samples=validation_samples,
        h_grid=validation_h_grid,
        half_width=beta2_half_width,
        spatial_points=beta2_spatial_points,
    )
    beta2_fft_maximum_absolute_error = float(
        np.max(np.abs(validation_fft - validation_exact))
    )
    print(
        "FFT validation maximum absolute error: "
        f"{beta2_fft_maximum_absolute_error:.3e}"
    )

beta2_rng = np.random.default_rng(beta2_seed)
beta2_uncorrected_indices = np.empty(beta2_M, dtype=int)
beta2_corrected_indices = np.empty(beta2_M, dtype=int)

for repetition in range(beta2_M):
    current_samples = beta2_rng.normal(size=beta2_n)
    current_similarity = beta2_similarity_matrix_fft(
        samples=current_samples,
        h_grid=beta2_h_grid,
        tau=beta2_tau,
        half_width=beta2_half_width,
        spatial_points=beta2_spatial_points,
    )

    current_stationary = doob_stationary_mass(current_similarity)
    beta2_uncorrected_indices[repetition] = int(
        np.argmax(current_stationary)
    )
    _, corrected_index, _ = toeplitz_corrected_choice(
        similarity=current_similarity,
        bandwidths=beta2_h_grid,
    )
    beta2_corrected_indices[repetition] = corrected_index

    if (repetition + 1) % 5 == 0 or repetition + 1 == beta2_M:
        print(f"completed {repetition + 1}/{beta2_M} repetitions")

beta2_uncorrected_counts = np.bincount(
    beta2_uncorrected_indices,
    minlength=beta2_N,
)
beta2_corrected_counts = np.bincount(
    beta2_corrected_indices,
    minlength=beta2_N,
)

FFT validation maximum absolute error: 2.176e-14
completed 5/100 repetitions
completed 10/100 repetitions
completed 15/100 repetitions
completed 20/100 repetitions
completed 25/100 repetitions
completed 30/100 repetitions
completed 35/100 repetitions
completed 40/100 repetitions
completed 45/100 repetitions
completed 50/100 repetitions
completed 55/100 repetitions
completed 60/100 repetitions
completed 65/100 repetitions
completed 70/100 repetitions
completed 75/100 repetitions
completed 80/100 repetitions
completed 85/100 repetitions
completed 90/100 repetitions
completed 95/100 repetitions
completed 100/100 repetitions


In [8]:
beta2_tick_indices = np.unique(
    np.linspace(0, beta2_N - 1, 7, dtype=int)
)
beta2_tick_bandwidths = [
    f"{beta2_h_grid[index]:.3g}" for index in beta2_tick_indices
]

for label, counts, color in (
    ("Uncorrected Doob", beta2_uncorrected_counts, "#1f77b4"),
    ("Toeplitz-corrected", beta2_corrected_counts, "#ff7f0e"),
):
    nonzero_indices = np.flatnonzero(counts)
    figure = go.Figure(
        go.Bar(
            x=nonzero_indices,
            y=counts[nonzero_indices],
            customdata=beta2_h_grid[nonzero_indices],
            width=0.85,
            marker=dict(
                color=color,
                line=dict(color="white", width=0.5),
            ),
            hovertemplate=(
                "grid index=%{x}"
                "<br>bandwidth h=%{customdata:.6g}"
                "<br>frequency=%{y}<extra></extra>"
            ),
        )
    )
    figure.update_xaxes(
        title_text="selected bandwidth h (geometric grid)",
        range=[-0.5, beta2_N - 0.5],
        tickmode="array",
        tickvals=beta2_tick_indices,
        ticktext=beta2_tick_bandwidths,
    )
    figure.update_yaxes(
        title_text=f"frequency among {beta2_M} repetitions",
        rangemode="tozero",
        dtick=1,
    )
    figure.update_layout(
        title=(
            f"{label} beta-2 selections: n={beta2_n}, "
            f"N={beta2_N}, M={beta2_M}, tau={beta2_tau:g}"
        ),
        width=950,
        height=500,
        template="plotly_white",
        showlegend=False,
    )
    figure.show()

## 3. Beta-2 histograms for increasing sample size

This repeats the FFT-accelerated sweep for $n=100,200,\ldots,1000$ on one fixed geometric grid. Prefixes of the same maximum-size sample are reused within each repetition, and both methods operate on exactly the same beta-2 similarity matrix. Each panel overlays the uncorrected selection distribution in blue and the Toeplitz-corrected distribution with narrower orange bars. Dashed vertical lines mark the nearest grid indices to the geometric center and the Gaussian MISE-oracle bandwidth.

In [9]:
beta2_n_sweep_values = np.arange(100, 1001, 100, dtype=int)
beta2_n_sweep_N = 201
beta2_n_sweep_M = 500
beta2_n_sweep_h_min = beta2_h_min
beta2_n_sweep_h_max = beta2_h_max
beta2_n_sweep_tau = beta2_tau
beta2_n_sweep_seed = beta2_seed + 2

beta2_n_sweep_h_grid = np.geomspace(
    beta2_n_sweep_h_min,
    beta2_n_sweep_h_max,
    beta2_n_sweep_N,
)
beta2_n_sweep_center_h = np.sqrt(
    beta2_n_sweep_h_min * beta2_n_sweep_h_max
)
beta2_n_sweep_center_index = nearest_bandwidth_grid_index(
    beta2_n_sweep_center_h,
    beta2_n_sweep_h_grid,
)
beta2_n_sweep_oracle_bandwidths = {
    int(current_n): gaussian_mise_oracle_bandwidth(int(current_n))
    for current_n in beta2_n_sweep_values
}
beta2_n_sweep_oracle_indices = {
    int(current_n): nearest_bandwidth_grid_index(
        beta2_n_sweep_oracle_bandwidths[int(current_n)],
        beta2_n_sweep_h_grid,
    )
    for current_n in beta2_n_sweep_values
}

beta2_n_sweep_rng = np.random.default_rng(beta2_n_sweep_seed)
beta2_n_sweep_full_samples = beta2_n_sweep_rng.normal(
    size=(beta2_n_sweep_M, int(np.max(beta2_n_sweep_values)))
)
beta2_n_sweep_results = {}

for current_n in beta2_n_sweep_values:
    uncorrected_indices = np.empty(beta2_n_sweep_M, dtype=int)
    corrected_indices = np.empty(beta2_n_sweep_M, dtype=int)

    for repetition in range(beta2_n_sweep_M):
        current_samples = beta2_n_sweep_full_samples[
            repetition, : int(current_n)
        ]
        current_similarity = beta2_similarity_matrix_fft(
            samples=current_samples,
            h_grid=beta2_n_sweep_h_grid,
            tau=beta2_n_sweep_tau,
            half_width=beta2_half_width,
            spatial_points=beta2_spatial_points,
        )

        uncorrected_indices[repetition] = int(
            np.argmax(doob_stationary_mass(current_similarity))
        )
        _, corrected_index, _ = toeplitz_corrected_choice(
            similarity=current_similarity,
            bandwidths=beta2_n_sweep_h_grid,
        )
        corrected_indices[repetition] = corrected_index

    uncorrected_counts = np.bincount(
        uncorrected_indices,
        minlength=beta2_n_sweep_N,
    )
    corrected_counts = np.bincount(
        corrected_indices,
        minlength=beta2_n_sweep_N,
    )
    beta2_n_sweep_results[int(current_n)] = {
        "counts": uncorrected_counts,
        "relative_frequencies": (
            uncorrected_counts / beta2_n_sweep_M
        ),
        "uncorrected_indices": uncorrected_indices.copy(),
        "uncorrected_counts": uncorrected_counts,
        "uncorrected_relative_frequencies": (
            uncorrected_counts / beta2_n_sweep_M
        ),
        "corrected_indices": corrected_indices.copy(),
        "corrected_counts": corrected_counts,
        "corrected_relative_frequencies": (
            corrected_counts / beta2_n_sweep_M
        ),
        "oracle_bandwidth": beta2_n_sweep_oracle_bandwidths[
            int(current_n)
        ],
        "oracle_index": beta2_n_sweep_oracle_indices[int(current_n)],
    }
    print(
        f"completed n={current_n}: "
        f"same index={np.mean(uncorrected_indices == corrected_indices):.1%}"
    )

completed n=100: same index=0.0%
completed n=200: same index=0.0%
completed n=300: same index=0.0%
completed n=400: same index=0.0%
completed n=500: same index=0.0%
completed n=600: same index=0.0%
completed n=700: same index=0.0%
completed n=800: same index=0.0%
completed n=900: same index=0.0%
completed n=1000: same index=0.0%


In [10]:
beta2_n_rows = len(beta2_n_sweep_values)
beta2_n_ticks = np.unique(
    np.linspace(0, beta2_n_sweep_N - 1, 7, dtype=int)
)
beta2_n_tick_labels = [
    f"{beta2_n_sweep_h_grid[index]:.3g}" for index in beta2_n_ticks
]
beta2_n_max_frequency = max(
    max(
        float(result["uncorrected_relative_frequencies"].max()),
        float(result["corrected_relative_frequencies"].max()),
    )
    for result in beta2_n_sweep_results.values()
)
beta2_n_line_height = 1.05 * beta2_n_max_frequency

beta2_n_stacked = make_subplots(
    rows=beta2_n_rows,
    cols=1,
    shared_xaxes=True,
    subplot_titles=[f"n={value}" for value in beta2_n_sweep_values],
    vertical_spacing=0.012,
)
beta2_n_overlay = go.Figure()

for row, current_n in enumerate(beta2_n_sweep_values, start=1):
    result = beta2_n_sweep_results[int(current_n)]
    uncorrected_nonzero = np.flatnonzero(result["uncorrected_counts"])
    corrected_nonzero = np.flatnonzero(result["corrected_counts"])
    uncorrected_y = result[
        "uncorrected_relative_frequencies"
    ][uncorrected_nonzero]
    corrected_y = result[
        "corrected_relative_frequencies"
    ][corrected_nonzero]

    beta2_n_stacked.add_trace(
        go.Bar(
            x=uncorrected_nonzero,
            y=uncorrected_y,
            customdata=beta2_n_sweep_h_grid[uncorrected_nonzero],
            width=0.85,
            marker=dict(
                color="#1f77b4",
                line=dict(color="white", width=0.35),
            ),
            opacity=0.72,
            name="uncorrected Doob selections",
            legendgroup="beta2-uncorrected",
            hovertemplate=(
                f"uncorrected, n={current_n}"
                "<br>h=%{customdata:.6g}"
                "<br>relative frequency=%{y:.1%}<extra></extra>"
            ),
            showlegend=(row == 1),
        ),
        row=row,
        col=1,
    )
    beta2_n_stacked.add_trace(
        go.Bar(
            x=corrected_nonzero,
            y=corrected_y,
            customdata=beta2_n_sweep_h_grid[corrected_nonzero],
            width=0.45,
            marker=dict(
                color="#ff7f0e",
                line=dict(color="white", width=0.35),
            ),
            opacity=0.9,
            name="Toeplitz-corrected selections",
            legendgroup="beta2-corrected",
            hovertemplate=(
                f"Toeplitz-corrected, n={current_n}"
                "<br>h=%{customdata:.6g}"
                "<br>relative frequency=%{y:.1%}<extra></extra>"
            ),
            showlegend=(row == 1),
        ),
        row=row,
        col=1,
    )

    for x_index, name, color, legend_group, show_legend in (
        (
            beta2_n_sweep_center_index,
            "geometric center (nearest index)",
            "#555555",
            "beta2-center",
            row == 1,
        ),
        (
            result["oracle_index"],
            "Gaussian MISE oracle (nearest index)",
            "#d62728",
            "beta2-oracle",
            row == 1,
        ),
    ):
        beta2_n_stacked.add_trace(
            go.Scatter(
                x=[x_index, x_index],
                y=[0.0, beta2_n_line_height],
                mode="lines",
                line=dict(color=color, width=2, dash="dash"),
                name=name,
                legendgroup=legend_group,
                showlegend=show_legend,
            ),
            row=row,
            col=1,
        )
    beta2_n_stacked.update_yaxes(
        range=[0.0, beta2_n_line_height],
        tickformat=".0%",
        nticks=4,
        row=row,
        col=1,
    )

    color = f"hsl({360.0 * (row - 1) / beta2_n_rows:.1f},65%,45%)"
    beta2_n_overlay.add_trace(
        go.Bar(
            x=uncorrected_nonzero,
            y=uncorrected_y,
            customdata=beta2_n_sweep_h_grid[uncorrected_nonzero],
            width=0.85,
            marker_color=color,
            opacity=0.22,
            name=f"n={current_n}",
            hovertemplate=(
                f"uncorrected, n={current_n}"
                "<br>h=%{customdata:.6g}"
                "<br>relative frequency=%{y:.1%}<extra></extra>"
            ),
        )
    )

beta2_n_stacked.update_xaxes(
    range=[-0.5, beta2_n_sweep_N - 0.5],
    tickmode="array",
    tickvals=beta2_n_ticks,
    ticktext=beta2_n_tick_labels,
)
beta2_n_stacked.update_xaxes(
    title_text="selected bandwidth h (geometric grid)",
    row=beta2_n_rows,
    col=1,
)
beta2_n_stacked.update_layout(
    title=(
        f"FFT beta-2 selections for increasing n: "
        f"uncorrected (blue) and Toeplitz-corrected (orange), "
        f"N={beta2_n_sweep_N}, M={beta2_n_sweep_M}"
    ),
    width=1050,
    height=230 * beta2_n_rows + 170,
    template="plotly_white",
    barmode="overlay",
    legend=dict(orientation="h", x=0.5, xanchor="center", y=-0.025),
    margin=dict(t=100, b=160),
)
beta2_n_stacked.show()

beta2_n_overlay.update_xaxes(
    title_text="selected bandwidth h (geometric grid)",
    range=[-0.5, beta2_n_sweep_N - 0.5],
    tickmode="array",
    tickvals=beta2_n_ticks,
    ticktext=beta2_n_tick_labels,
)
beta2_n_overlay.update_yaxes(
    title_text="relative frequency",
    tickformat=".0%",
)
beta2_n_overlay.update_layout(
    title=(
        f"Overlay of uncorrected FFT beta-2 histograms: "
        f"N={beta2_n_sweep_N}, M={beta2_n_sweep_M}"
    ),
    width=1100,
    height=650,
    template="plotly_white",
    barmode="overlay",
    legend=dict(title="sample size"),
)
beta2_n_overlay.show()

## 4. Sample, exact-population, and first-order Perron vectors

This experiment compares three sum-normalized Perron vectors on the same bandwidth grid:

1. the sample vector obtained from the FFT beta-2 similarity matrix;
2. the exact population vector obtained by assembling the analytic Gaussian population matrix and solving its Perron eigenproblem numerically; and
3. the first-order perturbation formula

$$
v_i^{(1)}
=\frac{1}{N}
-\frac{\tau}{N^2}\left(r_i-\bar r\right),
\qquad
r_i=\sum_j D_{ij}^{\mathrm{pop}}.
$$

The implicit row-sum characterization determines the maximum of the first-order vector; it does not by itself provide the full exact Perron vector. All curves are displayed as $Nv_i$, so the horizontal value one corresponds to the uniform vector.

In [11]:
perron_comparison_n = 1000
perron_comparison_N = 191
perron_comparison_h_min = 0.01
perron_comparison_h_max = 2.0
perron_comparison_tau = 1.0
# Match the first repetition of the beta-2 sample-size sweep.
perron_comparison_seed = seed + 12
perron_comparison_half_width = beta2_half_width
perron_comparison_spatial_points = beta2_spatial_points

if perron_comparison_n < 1 or perron_comparison_N < 2:
    raise ValueError("Require n >= 1 and N >= 2.")
if not 0.0 < perron_comparison_h_min < perron_comparison_h_max:
    raise ValueError("Require 0 < h_min < h_max.")
if perron_comparison_tau < 0.0:
    raise ValueError("perron_comparison_tau must be nonnegative.")

perron_comparison_h_grid = np.geomspace(
    perron_comparison_h_min,
    perron_comparison_h_max,
    perron_comparison_N,
)
perron_comparison_rng = np.random.default_rng(perron_comparison_seed)
perron_comparison_samples = perron_comparison_rng.normal(
    size=perron_comparison_n
)

perron_comparison_sample_similarity = beta2_similarity_matrix_fft(
    samples=perron_comparison_samples,
    h_grid=perron_comparison_h_grid,
    tau=perron_comparison_tau,
    half_width=perron_comparison_half_width,
    spatial_points=perron_comparison_spatial_points,
)
perron_comparison_sample_vector = perron_vector(
    perron_comparison_sample_similarity
)

perron_comparison_population_divergence = (
    population_gaussian_beta2_divergence_matrix(
        h_grid=perron_comparison_h_grid,
        target_sigma=1.0,
    )
)
perron_comparison_population_similarity = np.exp(
    -perron_comparison_tau
    * perron_comparison_population_divergence
)
np.fill_diagonal(perron_comparison_population_similarity, 1.0)
perron_comparison_population_vector = perron_vector(
    perron_comparison_population_similarity
)

perron_comparison_first_order_vector = first_order_perron_vector(
    divergence_matrix=perron_comparison_population_divergence,
    tau=perron_comparison_tau,
)
perron_comparison_row_sums = (
    perron_comparison_population_divergence.sum(axis=1)
)
perron_comparison_implicit_index = int(
    np.argmin(perron_comparison_row_sums)
)

perron_comparison_vectors = {
    "sample FFT": perron_comparison_sample_vector,
    "exact population": perron_comparison_population_vector,
    "first-order population": perron_comparison_first_order_vector,
}
perron_comparison_selected_indices = {
    name: int(np.argmax(vector))
    for name, vector in perron_comparison_vectors.items()
}

for name, selected_index in perron_comparison_selected_indices.items():
    print(
        f"{name}: index={selected_index}, "
        f"h={perron_comparison_h_grid[selected_index]:.6g}"
    )
print(
    "implicit row-sum characterization: "
    f"index={perron_comparison_implicit_index}, "
    f"h={perron_comparison_h_grid[perron_comparison_implicit_index]:.6g}"
)
print(
    "maximum population first-order error: "
    f"{np.max(np.abs(perron_comparison_population_vector - perron_comparison_first_order_vector)):.3e}"
)

sample FFT: index=137, h=0.456209
exact population: index=138, h=0.46911
first-order population: index=138, h=0.46911
implicit row-sum characterization: index=138, h=0.46911
maximum population first-order error: 1.577e-06


In [12]:
perron_comparison_colors = {
    "sample FFT": "#1f77b4",
    "exact population": "#d62728",
    "first-order population": "#2ca02c",
}
perron_comparison_dashes = {
    "sample FFT": "solid",
    "exact population": "solid",
    "first-order population": "dash",
}

perron_comparison_figure = go.Figure()
for name, vector in perron_comparison_vectors.items():
    selected_index = perron_comparison_selected_indices[name]
    relative_vector = perron_comparison_N * vector
    perron_comparison_figure.add_trace(
        go.Scatter(
            x=perron_comparison_h_grid,
            y=relative_vector,
            customdata=vector,
            mode="lines",
            line=dict(
                color=perron_comparison_colors[name],
                width=2.5,
                dash=perron_comparison_dashes[name],
            ),
            name=name,
            hovertemplate=(
                f"{name}"
                "<br>h=%{x:.6g}"
                "<br>N v(h)=%{y:.6g}"
                "<br>v(h)=%{customdata:.8f}<extra></extra>"
            ),
        )
    )
    perron_comparison_figure.add_trace(
        go.Scatter(
            x=[perron_comparison_h_grid[selected_index]],
            y=[relative_vector[selected_index]],
            mode="markers",
            marker=dict(
                color=perron_comparison_colors[name],
                size=10,
                symbol="diamond",
                line=dict(color="white", width=1),
            ),
            name=f"{name} maximum",
            showlegend=False,
            hovertemplate=(
                f"{name} maximum"
                f"<br>index={selected_index}"
                "<br>h=%{x:.6g}"
                "<br>N v(h)=%{y:.6g}<extra></extra>"
            ),
        )
    )

perron_comparison_figure.add_hline(
    y=1.0,
    line=dict(color="#777777", width=1.5, dash="dot"),
    annotation_text="uniform vector",
    annotation_position="bottom right",
)
perron_comparison_figure.update_xaxes(
    title_text="bandwidth h (logarithmic scale)",
    type="log",
    range=[
        np.log10(perron_comparison_h_min),
        np.log10(perron_comparison_h_max),
    ],
)
perron_comparison_figure.update_yaxes(
    title_text="relative Perron component N v(h) (uniform = 1)",
)
perron_comparison_figure.update_layout(
    title=(
        f"Sample, exact-population, and first-order beta-2 Perron vectors: "
        f"n={perron_comparison_n}, N={perron_comparison_N}, "
        f"tau={perron_comparison_tau:g}"
    ),
    width=1100,
    height=650,
    template="plotly_white",
    legend=dict(
        orientation="h",
        x=0.5,
        xanchor="center",
        y=-0.18,
    ),
    margin=dict(b=130),
)
perron_comparison_figure.show()

## 5. Uncorrected versus Toeplitz-corrected selection distributions

This Monte Carlo experiment uses the same sample in each repetition for both methods. The upper panel selects the largest component of the uncorrected Doob stationary distribution. The lower panel applies the Toeplitz correction to the same FFT beta-2 similarity matrix. Aligned axes and a common vertical scale make differences between the two selection distributions directly comparable.

In [13]:
correction_comparison_n = 1000
correction_comparison_N = 191
correction_comparison_M = 100
correction_comparison_h_min = 0.01
correction_comparison_h_max = 2.0
correction_comparison_tau = 1.0
correction_comparison_seed = seed + 30
correction_comparison_half_width = beta2_half_width
correction_comparison_spatial_points = beta2_spatial_points

if correction_comparison_n < 1 or correction_comparison_M < 1:
    raise ValueError("n and M must be positive.")
if correction_comparison_N < 2:
    raise ValueError("N must be at least 2.")
if not 0.0 < correction_comparison_h_min < correction_comparison_h_max:
    raise ValueError("Require 0 < h_min < h_max.")
if correction_comparison_tau <= 0.0:
    raise ValueError("tau must be positive.")

correction_comparison_h_grid = np.geomspace(
    correction_comparison_h_min,
    correction_comparison_h_max,
    correction_comparison_N,
)
correction_comparison_rng = np.random.default_rng(
    correction_comparison_seed
)
correction_comparison_uncorrected_indices = np.empty(
    correction_comparison_M,
    dtype=int,
)
correction_comparison_corrected_indices = np.empty(
    correction_comparison_M,
    dtype=int,
)

for repetition in range(correction_comparison_M):
    current_samples = correction_comparison_rng.normal(
        size=correction_comparison_n
    )
    current_similarity = beta2_similarity_matrix_fft(
        samples=current_samples,
        h_grid=correction_comparison_h_grid,
        tau=correction_comparison_tau,
        half_width=correction_comparison_half_width,
        spatial_points=correction_comparison_spatial_points,
    )

    current_stationary = doob_stationary_mass(current_similarity)
    correction_comparison_uncorrected_indices[repetition] = int(
        np.argmax(current_stationary)
    )
    _, corrected_index, _ = toeplitz_corrected_choice(
        similarity=current_similarity,
        bandwidths=correction_comparison_h_grid,
    )
    correction_comparison_corrected_indices[repetition] = corrected_index

    if (repetition + 1) % 10 == 0:
        print(
            f"completed {repetition + 1}/"
            f"{correction_comparison_M} repetitions"
        )

correction_comparison_uncorrected_counts = np.bincount(
    correction_comparison_uncorrected_indices,
    minlength=correction_comparison_N,
)
correction_comparison_corrected_counts = np.bincount(
    correction_comparison_corrected_indices,
    minlength=correction_comparison_N,
)
correction_comparison_uncorrected_relative = (
    correction_comparison_uncorrected_counts / correction_comparison_M
)
correction_comparison_corrected_relative = (
    correction_comparison_corrected_counts / correction_comparison_M
)

for label, indices, counts in (
    (
        "uncorrected",
        correction_comparison_uncorrected_indices,
        correction_comparison_uncorrected_counts,
    ),
    (
        "Toeplitz-corrected",
        correction_comparison_corrected_indices,
        correction_comparison_corrected_counts,
    ),
):
    selected_bandwidths = correction_comparison_h_grid[indices]
    mode_index = int(np.argmax(counts))
    print(
        f"{label}: mean h={selected_bandwidths.mean():.6g}, "
        f"median h={np.median(selected_bandwidths):.6g}, "
        f"mode h={correction_comparison_h_grid[mode_index]:.6g}"
    )

correction_comparison_agreement = np.mean(
    correction_comparison_uncorrected_indices
    == correction_comparison_corrected_indices
)
print(
    "same selected grid index: "
    f"{correction_comparison_agreement:.1%}"
)

completed 10/100 repetitions
completed 20/100 repetitions
completed 30/100 repetitions
completed 40/100 repetitions
completed 50/100 repetitions
completed 60/100 repetitions
completed 70/100 repetitions
completed 80/100 repetitions
completed 90/100 repetitions
completed 100/100 repetitions
uncorrected: mean h=0.433531, median h=0.443663, mode h=0.431462
Toeplitz-corrected: mean h=0.379359, median h=0.637517, mode h=0.637517
same selected grid index: 0.0%


In [14]:
correction_comparison_tick_indices = np.unique(
    np.linspace(0, correction_comparison_N - 1, 7, dtype=int)
)
correction_comparison_tick_labels = [
    f"{correction_comparison_h_grid[index]:.3g}"
    for index in correction_comparison_tick_indices
]
correction_comparison_maximum_frequency = max(
    float(correction_comparison_uncorrected_relative.max()),
    float(correction_comparison_corrected_relative.max()),
)

correction_comparison_figure = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    subplot_titles=(
        "Uncorrected Doob selections",
        "Toeplitz-corrected selections",
    ),
    vertical_spacing=0.14,
)

for row, label, counts, relative_frequencies, color in (
    (
        1,
        "uncorrected",
        correction_comparison_uncorrected_counts,
        correction_comparison_uncorrected_relative,
        "#1f77b4",
    ),
    (
        2,
        "Toeplitz-corrected",
        correction_comparison_corrected_counts,
        correction_comparison_corrected_relative,
        "#ff7f0e",
    ),
):
    nonzero_indices = np.flatnonzero(counts)
    correction_comparison_figure.add_trace(
        go.Bar(
            x=nonzero_indices,
            y=relative_frequencies[nonzero_indices],
            customdata=correction_comparison_h_grid[nonzero_indices],
            width=0.85,
            marker=dict(
                color=color,
                line=dict(color="white", width=0.4),
            ),
            name=label,
            hovertemplate=(
                f"{label}"
                "<br>grid index=%{x}"
                "<br>bandwidth h=%{customdata:.6g}"
                "<br>relative frequency=%{y:.1%}<extra></extra>"
            ),
            showlegend=False,
        ),
        row=row,
        col=1,
    )
    correction_comparison_figure.update_yaxes(
        title_text="relative frequency",
        range=[0.0, 1.05 * correction_comparison_maximum_frequency],
        tickformat=".0%",
        row=row,
        col=1,
    )

correction_comparison_figure.update_xaxes(
    range=[-0.5, correction_comparison_N - 0.5],
    tickmode="array",
    tickvals=correction_comparison_tick_indices,
    ticktext=correction_comparison_tick_labels,
)
correction_comparison_figure.update_xaxes(
    title_text="selected bandwidth h (geometric grid)",
    row=2,
    col=1,
)
correction_comparison_figure.update_layout(
    title=(
        f"FFT beta-2 selection distributions: "
        f"n={correction_comparison_n}, "
        f"N={correction_comparison_N}, "
        f"M={correction_comparison_M}, "
        f"tau={correction_comparison_tau:g}"
    ),
    width=1050,
    height=800,
    template="plotly_white",
    margin=dict(t=110, b=90),
)
correction_comparison_figure.show()

## 6. Uncorrected selections on geometric and linear grids

This experiment compares uncorrected FFT beta-2 selections on two deterministic bandwidth grids with identical endpoints and the same number of candidates. The same Monte Carlo sample is used for both grids in each repetition.

The geometric grid assigns candidates uniformly in $\log h$, whereas the linear grid assigns candidates uniformly in $h$. Consequently, they represent different limiting measures over the bandwidth interval, and the Perron selector can legitimately change. Since their grid indices do not represent the same bandwidths, the selected bandwidths are aggregated into common logarithmic histogram bins for comparison.

In [15]:
grid_comparison_n = 1000
grid_comparison_N = 191
grid_comparison_M = 100
grid_comparison_h_min = 0.01
grid_comparison_h_max = 2.0
grid_comparison_tau = 1.0
grid_comparison_seed = seed + 40
grid_comparison_half_width = beta2_half_width
grid_comparison_spatial_points = beta2_spatial_points
grid_comparison_histogram_bins = 40

if grid_comparison_n < 1 or grid_comparison_M < 1:
    raise ValueError("n and M must be positive.")
if grid_comparison_N < 2:
    raise ValueError("N must be at least 2.")
if not 0.0 < grid_comparison_h_min < grid_comparison_h_max:
    raise ValueError("Require 0 < h_min < h_max.")
if grid_comparison_tau <= 0.0:
    raise ValueError("tau must be positive.")
if grid_comparison_histogram_bins < 2:
    raise ValueError("At least two common histogram bins are required.")

grid_comparison_geometric_grid = np.geomspace(
    grid_comparison_h_min,
    grid_comparison_h_max,
    grid_comparison_N,
)
grid_comparison_linear_grid = np.linspace(
    grid_comparison_h_min,
    grid_comparison_h_max,
    grid_comparison_N,
)
grid_comparison_geometric_indices = np.empty(
    grid_comparison_M,
    dtype=int,
)
grid_comparison_linear_indices = np.empty(
    grid_comparison_M,
    dtype=int,
)
grid_comparison_rng = np.random.default_rng(grid_comparison_seed)

for repetition in range(grid_comparison_M):
    current_samples = grid_comparison_rng.normal(size=grid_comparison_n)

    geometric_similarity = beta2_similarity_matrix_fft(
        samples=current_samples,
        h_grid=grid_comparison_geometric_grid,
        tau=grid_comparison_tau,
        half_width=grid_comparison_half_width,
        spatial_points=grid_comparison_spatial_points,
    )
    linear_similarity = beta2_similarity_matrix_fft(
        samples=current_samples,
        h_grid=grid_comparison_linear_grid,
        tau=grid_comparison_tau,
        half_width=grid_comparison_half_width,
        spatial_points=grid_comparison_spatial_points,
    )

    grid_comparison_geometric_indices[repetition] = int(
        np.argmax(doob_stationary_mass(geometric_similarity))
    )
    grid_comparison_linear_indices[repetition] = int(
        np.argmax(doob_stationary_mass(linear_similarity))
    )

    if (repetition + 1) % 10 == 0:
        print(
            f"completed {repetition + 1}/"
            f"{grid_comparison_M} repetitions"
        )

grid_comparison_geometric_bandwidths = (
    grid_comparison_geometric_grid[grid_comparison_geometric_indices]
)
grid_comparison_linear_bandwidths = (
    grid_comparison_linear_grid[grid_comparison_linear_indices]
)

for label, bandwidth_grid, selected_indices, selected_bandwidths in (
    (
        "geometric grid",
        grid_comparison_geometric_grid,
        grid_comparison_geometric_indices,
        grid_comparison_geometric_bandwidths,
    ),
    (
        "linear grid",
        grid_comparison_linear_grid,
        grid_comparison_linear_indices,
        grid_comparison_linear_bandwidths,
    ),
):
    selection_counts = np.bincount(
        selected_indices,
        minlength=grid_comparison_N,
    )
    mode_index = int(np.argmax(selection_counts))
    print(
        f"{label}: mean h={selected_bandwidths.mean():.6g}, "
        f"median h={np.median(selected_bandwidths):.6g}, "
        f"mode h={bandwidth_grid[mode_index]:.6g}"
    )

grid_comparison_absolute_differences = np.abs(
    grid_comparison_geometric_bandwidths
    - grid_comparison_linear_bandwidths
)
print(
    "paired mean absolute bandwidth difference: "
    f"{grid_comparison_absolute_differences.mean():.6g}"
)
print(
    "linear selection exceeds geometric selection: "
    f"{np.mean(grid_comparison_linear_bandwidths > grid_comparison_geometric_bandwidths):.1%}"
)

completed 10/100 repetitions
completed 20/100 repetitions
completed 30/100 repetitions
completed 40/100 repetitions
completed 50/100 repetitions
completed 60/100 repetitions
completed 70/100 repetitions
completed 80/100 repetitions
completed 90/100 repetitions
completed 100/100 repetitions
geometric grid: mean h=0.441697, median h=0.443663, mode h=0.443663
linear grid: mean h=0.992432, median h=0.994526, mode h=0.994526
paired mean absolute bandwidth difference: 0.550735
linear selection exceeds geometric selection: 100.0%


In [16]:
grid_comparison_log_edges = np.linspace(
    np.log10(grid_comparison_h_min),
    np.log10(grid_comparison_h_max),
    grid_comparison_histogram_bins + 1,
)
grid_comparison_log_centers = 0.5 * (
    grid_comparison_log_edges[:-1]
    + grid_comparison_log_edges[1:]
)
grid_comparison_log_width = (
    grid_comparison_log_edges[1] - grid_comparison_log_edges[0]
)

grid_comparison_geometric_histogram, _ = np.histogram(
    np.log10(grid_comparison_geometric_bandwidths),
    bins=grid_comparison_log_edges,
)
grid_comparison_linear_histogram, _ = np.histogram(
    np.log10(grid_comparison_linear_bandwidths),
    bins=grid_comparison_log_edges,
)
grid_comparison_geometric_relative = (
    grid_comparison_geometric_histogram / grid_comparison_M
)
grid_comparison_linear_relative = (
    grid_comparison_linear_histogram / grid_comparison_M
)
grid_comparison_maximum_frequency = max(
    float(grid_comparison_geometric_relative.max()),
    float(grid_comparison_linear_relative.max()),
)

grid_comparison_figure = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    subplot_titles=(
        "Uncorrected selections: geometric grid",
        "Uncorrected selections: linear grid",
    ),
    vertical_spacing=0.14,
)

for row, label, relative_frequencies, color in (
    (
        1,
        "geometric grid",
        grid_comparison_geometric_relative,
        "#1f77b4",
    ),
    (
        2,
        "linear grid",
        grid_comparison_linear_relative,
        "#9467bd",
    ),
):
    nonzero_bins = np.flatnonzero(relative_frequencies)
    grid_comparison_figure.add_trace(
        go.Bar(
            x=grid_comparison_log_centers[nonzero_bins],
            y=relative_frequencies[nonzero_bins],
            customdata=(
                10.0 ** grid_comparison_log_centers[nonzero_bins]
            ),
            width=0.9 * grid_comparison_log_width,
            marker=dict(
                color=color,
                line=dict(color="white", width=0.5),
            ),
            name=label,
            showlegend=False,
            hovertemplate=(
                f"{label}"
                "<br>bin center h=%{customdata:.6g}"
                "<br>relative frequency=%{y:.1%}<extra></extra>"
            ),
        ),
        row=row,
        col=1,
    )
    grid_comparison_figure.update_yaxes(
        title_text="relative frequency",
        range=[0.0, 1.05 * grid_comparison_maximum_frequency],
        tickformat=".0%",
        row=row,
        col=1,
    )

grid_comparison_tick_positions = np.linspace(
    np.log10(grid_comparison_h_min),
    np.log10(grid_comparison_h_max),
    7,
)
grid_comparison_tick_labels = [
    f"{10.0**position:.3g}"
    for position in grid_comparison_tick_positions
]
grid_comparison_figure.update_xaxes(
    range=[
        np.log10(grid_comparison_h_min),
        np.log10(grid_comparison_h_max),
    ],
    tickmode="array",
    tickvals=grid_comparison_tick_positions,
    ticktext=grid_comparison_tick_labels,
)
grid_comparison_figure.update_xaxes(
    title_text="selected bandwidth h (common logarithmic bins)",
    row=2,
    col=1,
)
grid_comparison_figure.update_layout(
    title=(
        f"FFT beta-2 uncorrected selections by grid measure: "
        f"n={grid_comparison_n}, N={grid_comparison_N}, "
        f"M={grid_comparison_M}, tau={grid_comparison_tau:g}"
    ),
    width=1050,
    height=800,
    template="plotly_white",
    margin=dict(t=110, b=90),
)
grid_comparison_figure.show()